# FRInGe Demo Notebook (Fisher–Rao Integrated Gradients)

This notebook runs a single-image FRInGe attribution, logs diagnostics, and produces the **geometry plot** (Riemannian vs Euclidean per-step distance) plus optional attribution visualizations.

**Assumptions**
- You are running this notebook from the repository root (so imports like `from FisherRaoIG...` work).
- Your image exists at `../examples/n01580077_jay.JPEG` (edit the path if needed).


In [ ]:
# If you're in a fresh environment, you may need:
# !pip install torch torchvision numpy matplotlib tqdm

import os
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

from FisherRaoIG.plot_utils import (
    set_icml_matplotlib_style,
    visualize_geometry,
    visualize_attr_triplet,
)
from FisherRaoIG.plot_intermediates import visualize_steps_prev_delta

from metrics.MetricsWrapper import MetricsWrapper
from utils import denormalize_image, load_model, load_image

# Import your class + logging config from your module.
# If the class is defined in a script instead, paste it into a cell or
# replace this import with the right module path.
from FisherRaoIG.BiharmonicFisherRaoIntegratedGradients import FisherRaoIntegratedGradients, LogConfig

## 1) Load model + image

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model, transformation = load_model("resnet18", device=device)

# Edit if needed
img_path = "./examples/n01580077_jay.JPEG"
assert os.path.exists(img_path), f"Missing image: {img_path}"

img = load_image(img_path, transformation, device=device)
batch = img  # (1,C,H,W)

with torch.no_grad():
    logits0 = model(batch)
    target_idx = logits0.argmax(dim=1)

print("Target index:", target_idx.item())


## 2) Run FRInGe attribution

Tune the hyperparameters below to explore stability and geometry.

In [ ]:
def model_forward_batch(x, target):
    return model(x).gather(1, target[:, None]).squeeze(1)

ig = FisherRaoIntegratedGradients(
    model=model,
    model_forward=model_forward_batch,
    target_idx=target_idx,
    use_exact_jvp=True,
)

cfg = LogConfig(
    enabled=True,
    to_cpu_numpy=True,
    store_viz=True,
    plot_viz=False,  # True = show intermediate panels during descent
    viz_every=3,
)

attributions, info = ig.attribute(
    batch,
    # core loop
    max_correction_steps=1,
    cg_max_iter=20,
    fisher=True,
    # trust region
    eta_base=10,
    delta_euc=5,
    kl_target=0.001,
    # damping / regularization
    gamma=0.01,
    alpha_start=0.001,
    alpha_end=0.001,
    diag_floor=0.01,
    clamp_max_diag=100,
    clamp_min_diag=0.001,
    gauss_blur_sigma_diag=2.0,
    gauss_blur_kernel_diag=5,
    # logging
    log_cfg=cfg,
    denormalize_image=denormalize_image,
    verbose=True,
)

print("Returned attribution shape:", tuple(attributions.shape))
print("Logged keys (sample):", list(info.keys())[:15], "...")


## 3) Geometry plot (Riemannian vs Euclidean per-step distance)

This is the key diagnostic to understand how your **Euclidean trust region** interacts with the **Fisher pullback geometry**.

In [ ]:
delta_euclidean = np.asarray(info["delta_euclidean"]).squeeze()
delta_riemann = np.asarray(info["delta_riemann"]).squeeze()
entropy = np.asarray(info["entropy"]).squeeze()

num_classes = logits0.shape[-1]
entropy = entropy / np.log(num_classes)

T = len(delta_euclidean)
steps = np.arange(T, dtype=float) / max(T, 1)

set_icml_matplotlib_style(
    base_fontsize=13,
    tick_fontsize=11,
    legend_fontsize=8,
    title_fontsize=14,
    line_width=2.0,
)

visualize_geometry(
    delta_riemann,
    delta_euclidean,
    steps,
    entropy=entropy,
    title="FRInGe Geometry (Euclidean TR)\nRiemannian distance vs Euclidean distance increment per step",
)


## 4) Optional: attribution visualization

In [ ]:
visualize_attr_triplet(denormalize_image(batch), attributions)


## 5) Optional: intermediate steps panel

Requires `store_viz=True` in `LogConfig`.

In [ ]:
if "intermediate_x" in info and len(info["intermediate_x"]) > 0:
    T = len(info["intermediate_x"])
    idxs = [0, int(0.1*T), int(0.2*T), int(0.35*T), T-1]
    idxs = sorted(set(max(0, min(T-1, i)) for i in idxs))
    visualize_steps_prev_delta(info, steps=idxs, C=num_classes)
else:
    print("No intermediate viz stored. Set LogConfig(store_viz=True).")


## 6) Optional: save logs


In [ ]:
save_path = "fisher_rao_info_with_euclidean_TR.pkl"
with open(save_path, "wb") as f:
    pickle.dump(info, f)
print("Saved:", save_path)


## 7) Optional: run metrics

Requires your `MetricsWrapper` and metric implementations to be available.

In [ ]:
run_metrics = True
if run_metrics:
    metrics = MetricsWrapper(ig, model)
    scores = metrics.extract_insertion_deletion_auc(batch, attributions)
    print(f"Insertion AUC: {scores['insertion_auc']:.4f}")
    print(f"Deletion AUC:  {scores['deletion_auc']:.4f}")

    mas_scores = metrics.extract_mas_score(batch, attributions)
    print(f"MAS Ins AUC:   {mas_scores['insertion']:.4f}")
    print(f"MAS Del AUC:   {mas_scores['deletion']:.4f}")

    infidelity_score = metrics.extract_infidelity_score(
        batch,
        attributions,
        target_idx=target_idx,
        n_perturbations=50,
        noise_scale=0.02,
    )
    print(infidelity_score.squeeze().item())
else:
    print("Metrics skipped. Set run_metrics=True to enable.")
